In [1]:
"""
Leave-one-style-out ActionStyle mats for GZSDA (ActionStyleDataset_v2).

Same filtering / seed as create-dataset.ipynb, then:

  - one data file per single style (target domains)
  - one pooled file per held-out style (source = all styles except that one)
  - one split file with 14 domain slots:
        0..6  = all_but_<style>  (concat of the other 6 styles' flags)
        7..13 = <style>          (same per-style train/test flags as v1)

Run from gzsda/data/ with inpDataset.npz in this directory.
"""


"\nLeave-one-style-out ActionStyle mats for GZSDA (ActionStyleDataset_v2).\n\nSame filtering / seed as create-dataset.ipynb, then:\n\n  - one data file per single style (target domains)\n  - one pooled file per held-out style (source = all styles except that one)\n  - one split file with 14 domain slots:\n        0..6  = all_but_<style>  (concat of the other 6 styles' flags)\n        7..13 = <style>          (same per-style train/test flags as v1)\n\nRun from gzsda/data/ with inpDataset.npz in this directory.\n"

In [2]:
import os
from collections import Counter

import numpy as np
import scipy.io


/tmp/ipykernel_23139/321693741.py:5: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.1)
  import scipy.io


In [3]:
# ----------------------------------------------------------------------
# Config -- tweak these as needed
# ----------------------------------------------------------------------
NPZ_PATH = "./inpDataset.npz"
OUT_DIR = "./ActionStyleDataset_v2/"


In [4]:
STYLE_LABELS = [
    "angry", "childlike", "depressed", "neutral",
    "old", "proud", "strutting",
]

PREFIX = "ActionStyle-"
SUFFIX = "-clip.mat"
FEATURE_KEY = "clip_features"
SPLIT_FILE_NAME = "instanceSplit_actionStyle_v2_unseen2.mat"

SOURCE_DOMAINS = [f"all_but_{s}" for s in STYLE_LABELS]
DOMAIN_SET = SOURCE_DOMAINS + STYLE_LABELS
PAIRS = [(i, i + len(STYLE_LABELS)) for i in range(len(STYLE_LABELS))]


In [5]:
ACTION_LABELS = ["punch", "jump", "kick", "walk"]

ACTION_FULL_NAMES = [
    "walk_s1", "walk_s2",
    "walk_lturn_s1", "walk_rturn_s1", "walk_lturn_l1", "walk_lturn_l2",
    "walk_l1", "walk_l2",
    "walk_rturn_s2", "walk_lturn_s2", "walk_rturn_l1", "walk_lturn_l3",
    "run", "run_lturn", "run_rturn",
    "jump_1", "jump_2",
    "punch_r", "punch_l", "punch_qr", "punch_ql",
    "kick_l", "kick_r",
    "trans_jump2walk", "trans_walk2jump", "trans_punch2kick",
    "trans_walk2punch", "trans_run2jump",
]


In [6]:
NUM_TRIALS = 6
NUM_UNSEEN_ACTIONS = 2          # held out per trial, same across all domains
TRAIN_RATIO = 0.7               # of the *seen*-class instances per domain
SEED = 42

# ----------------------------------------------------------------------
rng = np.random.default_rng(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

data = np.load(NPZ_PATH, allow_pickle=True)
x = data["clip_arrays"]            # (537, 1, 512)
names = data["names"]              # e.g. "Abe_childlike_05_001"


In [7]:
styles = np.array([n.split("_")[1] for n in names])
action_idx = np.array([n.split("_")[2] for n in names])
actions = np.array(
    [ACTION_FULL_NAMES[int(i) - 1].split("_")[0] for i in action_idx]
)

# drop transition clips (trans_*), keep every style this time
keep = actions != "trans"
x, styles, actions = x[keep], styles[keep], actions[keep]

# we used sampleing and it hurts differences between walk and run
# so we ignore run class
keep = actions != "run"
x, styles, actions = x[keep], styles[keep], actions[keep]

# balance dataset
keep = (rng.random((actions != "walk").shape) > 0.7) + (actions != "walk")
x, styles, actions = x[keep], styles[keep], actions[keep]

keep = styles != "sexy"
x, styles, actions = x[keep], styles[keep], actions[keep]


In [8]:
action_label_to_id = {label: i for i, label in enumerate(ACTION_LABELS)}
action_ids = np.array([action_label_to_id[a] for a in actions])
action_label_to_id


{'punch': 0, 'jump': 1, 'kick': 2, 'walk': 3}

In [9]:
feat = x.reshape(x.shape[0], -1).astype(np.float32)  # (N, D)


In [10]:
print("Per-style sample counts:")
for style in STYLE_LABELS:
    mask = styles == style
    feat_s = feat[mask]
    labels_s = action_ids[mask]

    mat_dict = {
        "labels": labels_s.reshape(1, -1).astype(np.int64),
        FEATURE_KEY: feat_s.reshape(feat_s.shape[0], feat_s.shape[1], 1, 1),
    }
    print(Counter(labels_s.tolist()).most_common())
    out_path = os.path.join(OUT_DIR, f"{PREFIX}{style}{SUFFIX}")
    scipy.io.savemat(out_path, mat_dict)
    print(f"  {style:10s} -> {feat_s.shape[0]:4d} samples  ({out_path})")


Per-style sample counts:
[(0, 35), (3, 34), (1, 24), (2, 24)]
  angry      ->  117 samples  (./ActionStyleDataset_v2/ActionStyle-angry-clip.mat)
[(0, 37), (2, 27), (3, 23), (1, 23)]
  childlike  ->  110 samples  (./ActionStyleDataset_v2/ActionStyle-childlike-clip.mat)
[(3, 46), (0, 41), (2, 28), (1, 18)]
  depressed  ->  133 samples  (./ActionStyleDataset_v2/ActionStyle-depressed-clip.mat)
[(0, 48), (3, 27), (1, 23), (2, 19)]
  neutral    ->  117 samples  (./ActionStyleDataset_v2/ActionStyle-neutral-clip.mat)
[(3, 32), (2, 27), (0, 23), (1, 13)]
  old        ->   95 samples  (./ActionStyleDataset_v2/ActionStyle-old-clip.mat)
[(3, 38), (0, 22), (1, 20), (2, 11)]
  proud      ->   91 samples  (./ActionStyleDataset_v2/ActionStyle-proud-clip.mat)
[(3, 47), (0, 34), (2, 23), (1, 7)]
  strutting  ->  111 samples  (./ActionStyleDataset_v2/ActionStyle-strutting-clip.mat)


In [11]:
print("Pooled all_but_* source files:")
for held_out in STYLE_LABELS:
    src_styles = [s for s in STYLE_LABELS if s != held_out]
    feat_parts = [feat[styles == s] for s in src_styles]
    label_parts = [action_ids[styles == s] for s in src_styles]
    feat_s = np.concatenate(feat_parts, axis=0)
    labels_s = np.concatenate(label_parts, axis=0)

    name = f"all_but_{held_out}"
    mat_dict = {
        "labels": labels_s.reshape(1, -1).astype(np.int64),
        FEATURE_KEY: feat_s.reshape(feat_s.shape[0], feat_s.shape[1], 1, 1),
    }
    out_path = os.path.join(OUT_DIR, f"{PREFIX}{name}{SUFFIX}")
    scipy.io.savemat(out_path, mat_dict)
    print(
        f"  {name:22s} -> {feat_s.shape[0]:4d} samples  "
        f"(from {', '.join(src_styles)})"
    )


Pooled all_but_* source files:
  all_but_angry          ->  657 samples  (from childlike, depressed, neutral, old, proud, strutting)
  all_but_childlike      ->  664 samples  (from angry, depressed, neutral, old, proud, strutting)
  all_but_depressed      ->  641 samples  (from angry, childlike, neutral, old, proud, strutting)
  all_but_neutral        ->  657 samples  (from angry, childlike, depressed, old, proud, strutting)
  all_but_old            ->  679 samples  (from angry, childlike, depressed, neutral, proud, strutting)
  all_but_proud          ->  683 samples  (from angry, childlike, depressed, neutral, old, strutting)
  all_but_strutting      ->  663 samples  (from angry, childlike, depressed, neutral, old, proud)


In [12]:
# ---- build the split file ----
# Draw per-style flags first (same RNG stream as create-dataset.ipynb),
# then copy them into target slots 7..13 and concat into source slots 0..6.
num_styles = len(STYLE_LABELS)
num_domains = 2 * num_styles
num_classes = len(ACTION_LABELS)

splitFlag_cells = np.empty((1, NUM_TRIALS), dtype=object)
unseenClass_cells = np.empty((1, NUM_TRIALS), dtype=object)


In [13]:
used_list = []
for t in range(NUM_TRIALS):
    unseen_action_ids = rng.choice(num_classes, size=NUM_UNSEEN_ACTIONS, replace=False)
    unseen_action_ids.sort()
    while True:
        rejected = False
        for arr in used_list:
            if np.all(unseen_action_ids == arr):
                unseen_action_ids = rng.choice(num_classes, size=NUM_UNSEEN_ACTIONS, replace=False)
                unseen_action_ids.sort()
                rejected = True
        if not rejected:
            used_list.append(unseen_action_ids)
            break
    print(used_list)

    unseen_indicator = np.zeros(num_classes, dtype=np.uint8)
    unseen_indicator[unseen_action_ids] = 1

    style_flags = {}
    for style in STYLE_LABELS:
        mask = styles == style
        labels_s = action_ids[mask]
        n = labels_s.shape[0]

        flag = np.zeros(n, dtype=np.uint8)
        is_unseen = np.isin(labels_s, unseen_action_ids)

        # unseen-class instances are never trained on -> test only
        flag[is_unseen] = 2

        # seen-class instances: split train/test per TRAIN_RATIO
        seen_idx = np.where(~is_unseen)[0]
        rng.shuffle(seen_idx)
        n_train = int(len(seen_idx) * TRAIN_RATIO)
        flag[seen_idx[:n_train]] = 1
        flag[seen_idx[n_train:]] = 2

        style_flags[style] = flag

    domain_splitFlags = np.empty((1, num_domains), dtype=object)
    domain_unseen = np.empty((1, num_domains), dtype=object)
    unseen_row = unseen_indicator.reshape(1, -1)

    for i, held_out in enumerate(STYLE_LABELS):
        pooled = np.concatenate(
            [style_flags[s] for s in STYLE_LABELS if s != held_out]
        )
        domain_splitFlags[0, i] = pooled.reshape(1, -1)
        domain_unseen[0, i] = unseen_row
        domain_splitFlags[0, i + num_styles] = style_flags[held_out].reshape(1, -1)
        domain_unseen[0, i + num_styles] = unseen_row

    splitFlag_cells[0, t] = domain_splitFlags
    unseenClass_cells[0, t] = domain_unseen


[array([0, 2])]
[array([0, 2]), array([1, 3])]
[array([0, 2]), array([1, 3]), array([2, 3])]
[array([0, 2]), array([1, 3]), array([2, 3]), array([0, 3])]
[array([0, 2]), array([1, 3]), array([2, 3]), array([0, 3]), array([0, 1])]
[array([0, 2]), array([1, 3]), array([2, 3]), array([0, 3]), array([0, 1]), array([1, 2])]


In [14]:
split_path = os.path.join(OUT_DIR, SPLIT_FILE_NAME)
scipy.io.savemat(
    split_path,
    {
        "targetDomain_splitFlag": splitFlag_cells,
        "targetDomain_unseenClass": unseenClass_cells,
    },
)
print(f"\nwrote split file -> {split_path}")

print(
    "\n--- use with TUPL / BaseTwoModalDataset ---\n"
    f"DOMAIN_SET = {DOMAIN_SET}\n"
    f"PAIRS = {PAIRS}  # all_but_X -> X\n"
    f"DATA_DIR = '{OUT_DIR}'\n"
    "DATASET_DETAILS = {\n"
    f"    'prefix': '{PREFIX}',\n"
    f"    'suffix': '{SUFFIX}',\n"
    f"    'resnet_feature': '{FEATURE_KEY}',\n"
    f"    'split_file_name': '{SPLIT_FILE_NAME}',\n"
    "}\n"
)



wrote split file -> ./ActionStyleDataset_v2/instanceSplit_actionStyle_v2_unseen2.mat

--- use with TUPL / BaseTwoModalDataset ---
DOMAIN_SET = ['all_but_angry', 'all_but_childlike', 'all_but_depressed', 'all_but_neutral', 'all_but_old', 'all_but_proud', 'all_but_strutting', 'angry', 'childlike', 'depressed', 'neutral', 'old', 'proud', 'strutting']
PAIRS = [(0, 7), (1, 8), (2, 9), (3, 10), (4, 11), (5, 12), (6, 13)]  # all_but_X -> X
DATA_DIR = './ActionStyleDataset_v2/'
DATASET_DETAILS = {
    'prefix': 'ActionStyle-',
    'suffix': '-clip.mat',
    'resnet_feature': 'clip_features',
    'split_file_name': 'instanceSplit_actionStyle_v2_unseen2.mat',
}

